# 02 — Preprocessing


Goals:
- Define predictors and target
- Decide which features to retain/exclude
- Separate numeric and categorical columns
- Create train/test split
- Build preprocessing pipeline


## 1. Library imports

Import the required Python libraries for data manipulation, dataset splitting,
numerical scaling, categorical encoding, missing-value imputation, and building
a reusable preprocessing pipeline.

In [2]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

## 2. Data load and Data Exploration

Load the Bank Marketing dataset (`bank-full.csv`) and perform an initial
exploration to understand its structure i.e. No. of observations, input features, and target.

In [ ]:
df = pd.read_csv("../data/bank-full.csv", sep=";")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Define Predictors and Target

Define the target variable (`y`) and encode into binary values (yes = 1, no = 0) and select the predictor variables (`X`).
Exclude five features (`default`, `day`, `duration`, `pdays`, and `poutcome`), retaining 11 predictors for model development.


## Feature Inclusion and Exclusion Decisions

The Bank Marketing dataset contains 16 input features and one target variable (`y`). For this project, 11 predictors are selected, and 5 are excluded.

**Included Features (11):**
- Numerical: `age`, `balance`, `campaign`, `previous`
- Categorical: `job`, `marital`, `education`, `housing`, `loan`, `contact`, `month`

**Excluded Features (5):**
- `duration`: Excluded to prevent data leakage because call duration is only known after the call.
- `default`: Excluded because the feature is highly imbalanced and may provide limited predictive information.
- `day`: Excluded to avoid relying on potentially unstable campaign scheduling patterns.
- `pdays`: Excluded from the initial model because its special value (`-1`) requires additional handling.
- `poutcome`: Excluded to simplify the initial model, although previous campaign outcomes may contain useful predictive information.

These are preliminary feature-selection decisions. Their impact on model performance has not yet been evaluated and can be revisited during model evaluation.

The selected features will be used to predict whether a customer subscribes to a term deposit (`y = 1`) or not (`y = 0`).

In [ ]:
# Define target variable
y = df["y"].map({"yes": 1, "no": 0})

# Exclude selected features
features_to_exclude = [
    "default", "day", "duration", "pdays", "poutcome"
]

# Define predictors
X = df.drop(columns=["y"] + features_to_exclude)

print("\nSelected predictors:", X.columns.tolist())


print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget distribution (%):")
print(y.value_counts(normalize=True) * 100)

## 4. Identify Numerical and Categorical Features

Identify numerical and categorical predictors to apply appropriate preprocessing techniques: scaling for numerical features and encoding (one-hot) for categorical features.

In [ ]:
# Identify numerical features
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Identify categorical features
categorical_features = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

## 5. Data Quality Checks and Handling Missing and Unknown Values

Check for actual missing values (NaN) and "unknown" entries in the selected predictors. Retain "unknown" as a valid categorical value and handle actual missing values using imputation in the preprocessing pipeline. Also, check any invalid numerical values, and duplicate records.

In [ ]:
# Check missing values
print("Missing values:")
print(X.isnull().sum())

# Check unknown values in categorical features
print("\nUnknown values:")
print((X[categorical_features] == "unknown").sum())

# Check numerical validity
print("\nInfinite numerical values:")
print(np.isinf(X[numerical_features].to_numpy(dtype=float)).sum())

# Check duplicate records
print("\nDuplicate records:", df.duplicated().sum())

## 6. Train/Test/Validation Split

Split the dataset into 70% training, 15% testing, and 15% validation sets using stratified sampling.

Stratified sampling preserves the original target class distribution (88.3% no and 11.7% yes) across all three sets, ensuring balanced representation of the original class proportions.

In [ ]:
# Split dataset into 70% training and 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split temporary data equally into testing and validation sets
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Validation features:", X_val.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)
print("Validation target:", y_val.shape)

## 7. Build Preprocessing Pipeline

Create preprocessing pipelines to handle missing values, standardize numerical features, and encode categorical features.

Fit the preprocessing pipeline only on the training data and apply the same transformations to the validation and testing sets to prevent data leakage.

In [ ]:
# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combine preprocessing pipelines
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit on training data and transform all three sets
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 8. Validate and Save Preprocessing Results

Validate the preprocessed training, validation, and testing datasets by checking their dimensions and ensuring there are no missing or infinite values.

Save the fitted preprocessing pipeline for reuse in model training and future predictions.

In [ ]:
# Validate processed datasets
for name, data in [
    ("Training", X_train_processed),
    ("Validation", X_val_processed),
    ("Testing", X_test_processed)
]:
    print(f"{name} shape:", data.shape)
    print(f"{name} missing values:", np.isnan(data).sum())
    print(f"{name} infinite values:", np.isinf(data).sum())

# Save fitted preprocessing pipeline
output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, output_dir / "preprocessor.joblib")

print("\nPreprocessing pipeline saved successfully.")